# 2025-11-24: Capaz que ya no sirve.

In [ ]:
import time
from struct import unpack
import sys
import pyvisa

def getBlockData(): # Función Obtener datos.
    global inBuffer
    global headerlen

    inBuffer = dso.read_bytes(10)
    length = len(inBuffer)
    headerlen = 2 + int(chr(inBuffer[1]))
    pkg_length = int(inBuffer[2:headerlen]) + headerlen # Bloque de datos #48008[8bytes][8000bytes raw data]

    print("Transfiriendo datos...")
    print('Longitud de paquete = %d' %pkg_length)
        
    pkg_length = pkg_length - length
    
    while True:
        if(pkg_length==0):
            break
        else:
            if(pkg_length > 100000):
                length = 100000
            else:
                length = pkg_length
            try:
                buf = dso.read_bytes(length)
            except:
                print('KeyboardInterrupt!')
                dso.closeIO()
                sys.exit(0)
            num = len(buf)
            inBuffer += buf
            pkg_length = pkg_length - num

if __name__ == '__main__':
    global inBuffer
    global headerlen
    
    rm = pyvisa.ResourceManager()
    dso = rm.open_resource('ASRL4::INSTR') # Iniciar comunicación con el instrumento.
    #dso.timeout = 5000
    dso.read_termination = '\n'
    dso.write_termination = '\n'
    idn = dso.query('*IDN?')
    print(idn) # Imprimir identificación del instrumento.

    ch = 1
    div = dso.query(':channel%d:scale?' %ch) # Obtener escala vertical.
    vdiv = float(div)
    print('Escala Vertical: %.2f [V/div]' %vdiv)

    dso.write(':acquire%d:state?' %ch)
    
    state = dso.read() # Leer estado de adquisición.
    if(state[0] == '1'):
        print('Forma de onda lista!')
    time.sleep(0.1)

    dso.write(":acquire%d:memory?" % ch) # Obtener forma de onda CH1 raw
    
    getBlockData()
    time.sleep(1)
    
    print(inBuffer[:headerlen]) # Imprimir cabecera de datos.
    dt = unpack('>f', inBuffer[headerlen : headerlen + 4]) # Periodo de muestreo.
    print("Periodo de muestreo = %.2e [s]" % dt) # Imprimir periodo de muestreo.
    waveform = unpack('>%sh' % (int(len(inBuffer[headerlen + 8:]) / 2)), inBuffer[headerlen + 8:])
    num = len(waveform)
    print('Cantidad de muestras = %d'%num) # Imprimir cantidad de muestras
    data = [0] * num
    for i in range(num):
        data[i] = waveform[i] * vdiv / 25
    
    print(data)
    dso.close()
    rm.close()

GW,GDS-1102A-U,GES170725,V1.14
Escala Vertical: 1.00 [V/div]
Forma de onda lista!
Transfiriendo datos...
Longitud de paquete = 8014
b'#48008'
Periodo de muestreo = 4.00e-07 [s]
Cantidad de muestras = 4000
[1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.92, 1.92, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.88, 1.88, 1.96, 1.96, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.92, 1.92, 1.92, 1.92, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.92, 1.92, 1.96, 1.96, 1.96, 1.

# 2025-11-24: Capaz que ya no sirve.

In [ ]:
import time
from struct import unpack
import sys
import pyvisa

def getBlockData(instrument): # Función Obtener datos.
    inBuffer = dso.read_bytes(10)
    length = len(inBuffer)
    headerlen = 2 + int(chr(inBuffer[1]))
    pkg_length = int(inBuffer[2:headerlen]) + headerlen # Bloque de datos #48008[8bytes][8000bytes raw data]
    
    print("Transfiriendo datos...")
    print('Longitud de paquete = %d' %pkg_length)
        
    pkg_length = pkg_length - length
    
    while True:
        if(pkg_length==0):
            break
        else:
            if(pkg_length > 100000):
                length = 100000
            else:
                length = pkg_length
            try:
                buf = instrument.read_bytes(length)
            except:
                print('KeyboardInterrupt!')
                instrument.closeIO()
                sys.exit(0)
            num = len(buf)
            inBuffer += buf
            pkg_length = pkg_length - num
    return inBuffer, headerlen

if __name__ == '__main__':
    inBuffer
    headerlen
    
    rm = pyvisa.ResourceManager()
    dso = rm.open_resource('ASRL4::INSTR') # Iniciar comunicación con el instrumento.
    #dso.timeout = 5000
    dso.read_termination = '\n'
    dso.write_termination = '\n'
    idn = dso.query('*IDN?')
    print(idn) # Imprimir identificación del instrumento.

    ch = 1
    div = dso.query(':channel%d:scale?' %ch) # Obtener escala vertical.
    vdiv = float(div)
    print('Escala Vertical: %.2f [V/div]' %vdiv)

    dso.write(':acquire%d:state?' %ch)
    
    state = dso.read() # Leer estado de adquisición.
    if(state[0] == '1'):
        print('Forma de onda lista!')
    time.sleep(0.1)

    dso.write(":acquire%d:memory?" % ch) # Obtener forma de onda CH1 raw
    
    getBlockData(dso)
    time.sleep(1)
    
    print(inBuffer[:headerlen]) # Imprimir cabecera de datos.
    dt = unpack('>f', inBuffer[headerlen : headerlen + 4]) # Periodo de muestreo.
    print("Periodo de muestreo = %.2e [s]" % dt) # Imprimir periodo de muestreo.
    waveform = unpack('>%sh' % (int(len(inBuffer[headerlen + 8:]) / 2)), inBuffer[headerlen + 8:])
    num = len(waveform)
    print('Cantidad de muestras = %d'%num) # Imprimir cantidad de muestras
    data = [0] * num
    for i in range(num):
        data[i] = waveform[i] * vdiv / 25
    
    print(data)
    dso.close()
    rm.close()

GW,GDS-1102A-U,GES170725,V1.14
Escala Vertical: 1.00 [V/div]
Forma de onda lista!
Transfiriendo datos...
Longitud de paquete = 8014
b'#48008'
Periodo de muestreo = 4.00e-07 [s]
Cantidad de muestras = 4000
[2.04, 2.04, 2.04, 2.0, 2.0, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 2.0, 2.0, 2.04, 2.04, 2.04, 2.04, 2.04, 2.04, 2.0, 2.0, 2.0, 2.0, 2.04, 2.04, 2.04, 2.04, 2.0, 2.0, 2.04, 2.04, 2.04, 2.04, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.96, 1.96, 2.0, 2.0, 1.96, 1.96, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.04, 2.04, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.04, 2.04, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.96, 1.96, 1.96, 1.96, 2.0, 2.0, 2.0, 2.0, 2.04, 2.04, 1.96, 1.96, 1.96, 1.96, 2.04, 2.04, 2.0, 2.0, 2.04, 2.04, 2.0, 2.0, 2.04, 2.04, 1.96, 1.96, 1.96, 1.96, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 2.04, 2.04, 1.96, 1.96, 2.0, 2.0, 2.04, 2.04, 1.

# 2025-11-24: Capaz que ya no sirve.

In [ ]:
def init_system():
    rm = pyvisa.ResourceManager()
    instrument_list = rm.list_resources()
    print("Instrumentos encontrados:", instrument_list)
    return rm

def init_instrument(manager, resource_name):
    try:
        dso = manager.open_resource(resource_name)
        dso.read_termination = '\n'
        dso.write_termination = '\n'
        idn = dso.query('*IDN?')
        print("Instrumento conectado satisfactoriamente:", idn)
    except Exception as e:
        print("Error al iniciar con el instrumento:", e)

def default_settings():
    try:
        dso.write('*RST')
        print("Se restableció el instrumento a la configuración de fábrica exitosamente.")
    except Exception as e:
        print("Error al restablecer el instrumento:", e)

def get_setting():
    try:
        current_setting = dso.query('*LRN?')
        print(f"Configuracion actual: {current_setting}")
    except Exception as e:
        print("Error al consultar configuración:", e)

def get_block_data(channel):
    try:
        v_div = get_vertical_scale(channel)
        dso.write(f':acquire{channel}:state?')
        state = dso.read()
        if(state[0] == '1'):
            time.sleep(0.1)
            dso.write(f":acquire{channel}:memory?")
            inBuffer = dso.read_bytes(10)
            length = len(inBuffer)
            headerlen = 2 + int(chr(inBuffer[1]))
            pkg_length = int(inBuffer[2:headerlen]) + headerlen
            pkg_length = pkg_length - length
            
            while True:
                if(pkg_length==0):
                    break
                else:
                    if(pkg_length > 100000):
                        length = 100000
                    else:
                        length = pkg_length
                    try:
                        buf = dso.read_bytes(length)
                    except:
                        print('KeyboardInterrupt!')
                        dso.closeIO()
                        sys.exit(0)
                    
                    num = len(buf)
                    inBuffer += buf
                    pkg_length = pkg_length - num
            waveform = unpack_waveform(inBuffer, headerlen, v_div)
            return waveform
        else:
            print('Error: Forma de onda aún no está lista.')
    except Exception as e:
        print("Error al obtener datos:", e)

def unpack_waveform(inBuffer, headerlen, vdiv):
    print(inBuffer[:headerlen])
    dt = unpack('>f', inBuffer[headerlen : headerlen + 4])
    print("Periodo de muestreo = %.2e [s]" % dt)
    waveform_raw = unpack('>%sh' % (int(len(inBuffer[headerlen + 8:]) / 2)), inBuffer[headerlen + 8:])
    num = len(waveform_raw)
    print(f'Cantidad de muestras = {num}')
    waveform = [0] * num
    for i in range(num):
        waveform[i] = waveform_raw[i] * vdiv / 25
    return waveform

def get_vertical_scale(channel):
    try:
        scale = dso.query(f':channel{channel}:scale?')
        v_scale = float(scale)
        print(f'Escala Vertical: {v_scale:.2f} [V/div]')
        return v_scale
    except Exception as e:
        print("Error al obtener la escala vertical:", e)

def set_vertical_scale(channel, value):
    try:
        dso.write(f':channel{channel}:scale {value}')
        v_scale = get_vertical_scale(channel)
        if v_scale == value:
            print(f'Escala vertical configurada a: {v_scale:.2f} [V/div]')
        else:
            print('No se pudo configurar la escala vertical.')
    except Exception as e:
        print("Error al obtener la escala vertical:", e)